## Step 1: Load and explore data
* Load the training datasets that contain the following information: Categorical, Quantitative, Connectome metrices, and training labels.
* Merge all data on common columns
* Identify and handle missing values, while standardise the formats of data

In [ ]:
# Import used libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load all training datasets
train_categorical = pd.read_excel("widsdatathon2025/TRAIN/TRAIN_CATEGORICAL_METADATA.xlsx")
train_connectome = pd.read_csv("widsdatathon2025/TRAIN/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES.csv")
train_quantitative = pd.read_excel("widsdatathon2025/TRAIN/TRAIN_QUANTITATIVE_METADATA.xlsx")
train_labels = pd.read_excel("widsdatathon2025/TRAIN/TRAINING_SOLUTIONS.xlsx")


In [ ]:
# Examine categorical data
print(train_categorical.head())
print(train_categorical.info())
print(train_categorical.shape)

In [ ]:
# fMRI data
print(train_connectome.head())
print(train_connectome.info())
print(train_connectome.shape)

In [ ]:
# Quantitative data
print(train_quantitative.head())
print(train_quantitative.info())
print(train_quantitative.shape)

In [ ]:
# Labels
print(train_labels.head())
print(train_labels.info())
print(train_labels.shape)

In [ ]:
# Merge datasets on the column 'participant_id'
train_data = pd.merge(train_categorical, train_quantitative, on='participant_id')
train_data = pd.merge(train_data, train_connectome, on='participant_id')
train_data = train_data.merge(train_labels, on='participant_id')

In [ ]:
# Identify missing values
missing_values = train_data.isnull().sum().sort_values(ascending=False)
print("Missing Values:\n", missing_values[missing_values > 0])

#### **Summary of the Dataset:**
**Categorical Metadata (10 columns, 1213 entries)**

* Contains data regarding study site information, as well as study subject's family background, and demographic data such as ethnicity
* The PreInt_Demos_Fam_Child_Ethnicity column contains 11 missing values to be handled

**Connectime Metrices (19901 columns, 1213 entries)**

* Contains fMRI Data that is recorded

**Quantitative Metadata (19 columns, 1213 entries)**

* Contains data regarding behavioural test scores
* The MRI_track_Age_at_Scan column contains 360 missing values to be handled

**Training Solutions (3 columns, 1213 entries)**

* Contains data regarding their final diagnosis of ADHD and study subject's gender

### Step 2: Preprocess the data and data visualisation

In [ ]:
# Handling missing data by filling missing values with the median
train_data.fillna(train_data.median(), inplace=True)

# Visualise missing data on train_data after handling it
sns.heatmap(train_data.isnull(), vmin=0, vmax=1, cmap="crest")
plt.title("Missing Data in train_data")
plt.show()

After the missing data is handled, it can be seen that all entries for all columns have no missing data from the heatmap above.

In [ ]:
# Split data into ADHD and no ADHD
adhd = train_data[train_data["ADHD_Outcome"] == 1]
no_adhd = train_data[train_data["ADHD_Outcome"] == 0]

# Split into Female and Male
female = train_data[train_data['Sex_F'] == 1]
male = train_data[train_data['Sex_F'] == 0]

# Print sample size
print(f"Sample Size with ADHD: {adhd.shape[0]}")
print(f"Sample Size without ADHD: {no_adhd.shape[0]}")
print(f"Sample Size of Women: {female.shape[0]}")
print(f"Sample Size of Men: {male.shape[0]}")

In [ ]:
# Visualise the ADHD Diagnosis Distribution by Gender
plt.figure(figsize=(8, 4))
ax = sns.countplot(x='ADHD_Outcome', hue='Sex_F', data=train_data, palette='Set1')

# Adding title and labels
plt.title("ADHD Diagnosis Distribution by Gender")
plt.xlabel("ADHD Diagnosis")
plt.ylabel("Number of Study Subjects")
plt.legend(title="Sex", labels=["Male (0)", "Female (1)"])

# Annotate the bars with percentage, each showing the percentage out of the whole dataset of the specific gender and diagnosis
total = len(train_data)
for p in ax.patches:
    height = p.get_height()
    percentage = (height / total) * 100
    ax.annotate(f'{percentage:.1f}%', (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', fontsize=10, color='black')

plt.show()

In [ ]:
numeric_data = train_data.select_dtypes(include=['number'])
sampled_data = numeric_data.sample(frac=0.10, random_state=42)
corr_matrix = sampled_data.corr(method='spearman')
corr_matrix = sampled_data.iloc[:, :30].corr(method='pearson')
plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, cmap='crest', vmax=1, vmin=-1,cbar=True, annot=False, square=False)
plt.title("Correlation Heatmap")
plt.show()

Following the analysis of the heat correlation map, it can be identified that there are several highly correlated identifiers:

* **Barratt_Barratt_P2_Occ:** Parent 2 Occupation
* **SDQ_SDQ_Conduct_Problems:** Conduct Problems Score
* **SDQ_SDQ_Difficulties_Total:** Total Difficulties Score
* **SDQ_SDQ_Emotional_Problems:** Emotional Problems Scale
* **SDQ_SDQ_Externalizing:** Externalizing Score
* **SDQ_SDQ_Generating_Impact:** Generating Impact Scores
* **SDQ_SDQ_Hyperactivity:** Hyperactivity Scale
* **SDQ_SDQ_Internalizing:** Internalizing Score
* **SDQ_SDQ_Peer_Problems:** Peer Problems Scale

In [ ]:
# Percentile Calculation
percentile_calc = ['SDQ_SDQ_Conduct_Problems', 
                   'SDQ_SDQ_Difficulties_Total', 
                   'SDQ_SDQ_Emotional_Problems', 
                   'SDQ_SDQ_Externalizing', 
                   'SDQ_SDQ_Generating_Impact', 
                   'SDQ_SDQ_Hyperactivity', 
                   'SDQ_SDQ_Internalizing', 
                   'SDQ_SDQ_Peer_Problems']
for column in percentile_calc: 
    q1 = np.percentile(train_data[column], 25)  # 25th percentile
    q2 = np.percentile(train_data[column], 50)  # 50th percentile (median)
    q3 = np.percentile(train_data[column], 75)  # 75th percentile
    q4 = np.max(train_data[column])  # 100th percentile (maximum)

    print(f"Percentile calculation for column {column}:")

    print(f"Q1 (25th percentile) : {q1}")
    print(f"Q2 (50th percentile - Median): {q2}")
    print(f"Q3 (75th percentile): {q3}")
    print(f"Q4 (100th percentile - Max): {q4}\n")

It can therefore be concluded that the following are the Q1-Q4 ranges of the self-assessment **Strength and Difficulties Questionnaire**:

| Column	                | 25% Percentile	| Median (50% Percentile)	| 75% Percentile	| 100% Percentile |
| ------------------------- | ----------------- | ------------------------- | ----------------- | --------------- |
| Conduct Problems Score	| 0	                | 0-2	                    | 2-3	            | 3-10            |
| Total Difficulties Score	| 0-7	            | 7-12	                    | 12-17	            | 17-34           |
| Emotional Problems Scale	| 0-1	            | 1-2	                    | 2-4	            | 4-10            |
| Externalizing Score	    | 0-4	            | 4-7	                    | 7-10	            | 10-20           |
| Generating Impact Scores	| 0-2	            | 2-4	                    | 4-6	            | 6-10            |
| Hyperactivity Score	    | 0-4	            | 4-6	                    | 6-8	            | 8-10            |
| Internalzing Score	    | 0-2	            | 2-4                    	| 4-7	            | 7-17            |
| Peer Problems Score	    | 0	                | 0-2                    	| 2-4            	| 4-9             |


### Step 3: Model selection

### Step 4: Train the model

### Step 5: Evaluate the model

### Step 6: Hyperparameter tuning